In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from loguru import logger

In [2]:
import os

# Go one level higher
os.chdir("..")

# Check current directory
print(os.getcwd())

/Users/rodrigo/football-data-analytics/football_xG


In [3]:
shots_df=pd.read_csv("xG/results/shots.csv")

In [4]:
shots_df.head(2)

,match_id,event_id,player_id,player_name,team_id,team_name,minute,second,shot_outcome,shot_body_part,shot_technique,x,y,dist_to_goal,angle_to_goal_rad,angle_to_goal_deg
0,NaN,943d24cf-3f41-41d0-81b9-c7237c374426,32119,Leandro Greco,226,Hellas Verona,9,41,Saved,Left Foot,Half Volley,94.5,54.5,29.334280,0.237022,13.580382
1,NaN,579178b1-c0c2-4e83-ac0f-5004a1c9190a,11502,Antonio Di Gaudio,1683,Carpi,11,3,Off T,Right Foot,Normal,100.6,34.5,20.164573,0.378200,21.669273


In [5]:
shots_df.iloc[0]

match_id                                              NaN
event_id             943d24cf-3f41-41d0-81b9-c7237c374426
player_id                                           32119
player_name                                 Leandro Greco
team_id                                               226
team_name                                   Hellas Verona
minute                                                  9
second                                                 41
shot_outcome                                        Saved
shot_body_part                                  Left Foot
shot_technique                                Half Volley
x                                                    94.5
y                                                    54.5
dist_to_goal                                     29.33428
angle_to_goal_rad                                0.237022
angle_to_goal_deg                               13.580382
Name: 0, dtype: object

## Do the EDA and see every output

In [6]:
# Configure loguru logger
logger.add("xg_eda_{time}.log", rotation="10 MB", level="DEBUG")

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [7]:

# ============================================================================
# STEP 1: LOAD DATA AND INITIAL INSPECTION
# ============================================================================

logger.info("="*80)
logger.info("STEP 1: Loading Data and Initial Inspection")
logger.info("="*80)

# Load your data
df = shots_df.copy()

logger.info(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
logger.debug(f"Columns: {list(df.columns)}")
logger.info(f"\nData types:\n{df.dtypes}")

2025-10-20 17:54:28.704 | INFO     | __main__:<module>:5 - ================================================================================
2025-10-20 17:54:28.705 | INFO     | __main__:<module>:6 - STEP 1: Loading Data and Initial Inspection
2025-10-20 17:54:28.706 | INFO     | __main__:<module>:7 - ================================================================================
2025-10-20 17:54:28.713 | INFO     | __main__:<module>:12 - Dataset shape: 88023 rows × 16 columns
2025-10-20 17:54:28.713 | DEBUG    | __main__:<module>:13 - Columns: ['match_id', 'event_id', 'player_id', 'player_name', 'team_id', 'team_name', 'minute', 'second', 'shot_outcome', 'shot_body_part', 'shot_technique', 'x', 'y', 'dist_to_goal', 'angle_to_goal_rad', 'angle_to_goal_deg']
2025-10-20 17:54:28.714 | INFO     | __main__:<module>:14 - 
Data types:
match_id             float64
event_id              object
player_id              int64
player_name           object
team_id                int64
team_name     

In [8]:
# ============================================================================
# STEP 2: MISSING VALUES ANALYSIS
# ============================================================================

logger.info("="*80)
logger.info("STEP 2: Analyzing Missing Values")
logger.info("="*80)

missing_stats = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_stats = missing_stats[missing_stats['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_stats) > 0:
    logger.warning(f"Found {len(missing_stats)} columns with missing values:")
    for _, row in missing_stats.iterrows():
        logger.warning(f"  - {row['Column']}: {row['Missing_Count']} ({row['Missing_Percentage']}%)")
else:
    logger.success("No missing values found!")

2025-10-20 17:54:31.205 | INFO     | __main__:<module>:5 - ================================================================================
2025-10-20 17:54:31.206 | INFO     | __main__:<module>:6 - STEP 2: Analyzing Missing Values
2025-10-20 17:54:31.207 | INFO     | __main__:<module>:7 - ================================================================================
2025-10-20 17:54:31.239 | WARNING  | __main__:<module>:17 - Found 1 columns with missing values:
2025-10-20 17:54:31.239 | WARNING  | __main__:<module>:19 -   - match_id: 88023 (100.0%)


In [9]:
# ============================================================================
# STEP 3: COORDINATE VALUES VALIDATION (X, Y)
# ============================================================================

logger.info("="*80)
logger.info("STEP 3: Validating Shot Coordinates")
logger.info("="*80)

# Football pitch dimensions (typical: 105m x 68m, coordinates usually 0-105 and 0-68)
logger.info("Checking X coordinates (expected range: ~0-120)")
logger.debug(f"X stats - Min: {df['x'].min():.2f}, Max: {df['x'].max():.2f}, Mean: {df['x'].mean():.2f}, Median: {df['x'].median():.2f}")

x_outliers = df[(df['x'] < 0) | (df['x'] > 120)]
if len(x_outliers) > 0:
    logger.warning(f"Found {len(x_outliers)} shots with suspicious X coordinates (outside 0-120 range)")
    logger.debug(f"Sample of X outliers:\n{x_outliers[['player_name', 'team_name', 'x', 'y']].head()}")
else:
    logger.success("All X coordinates are within reasonable range (0-120)")

logger.info("Checking Y coordinates (expected range: ~0-80)")
logger.debug(f"Y stats - Min: {df['y'].min():.2f}, Max: {df['y'].max():.2f}, Mean: {df['y'].mean():.2f}, Median: {df['y'].median():.2f}")

y_outliers = df[(df['y'] < 0) | (df['y'] > 80)]
if len(y_outliers) > 0:
    logger.warning(f"Found {len(y_outliers)} shots with suspicious Y coordinates (outside 0-80 range)")
    logger.debug(f"Sample of Y outliers:\n{y_outliers[['player_name', 'team_name', 'x', 'y']].head()}")
else:
    logger.success("All Y coordinates are within reasonable range (0-80)")

2025-10-20 17:54:42.670 | INFO     | __main__:<module>:5 - ================================================================================
2025-10-20 17:54:42.671 | INFO     | __main__:<module>:6 - STEP 3: Validating Shot Coordinates
2025-10-20 17:54:42.671 | INFO     | __main__:<module>:7 - ================================================================================
2025-10-20 17:54:42.672 | INFO     | __main__:<module>:10 - Checking X coordinates (expected range: ~0-120)
2025-10-20 17:54:42.677 | DEBUG    | __main__:<module>:11 - X stats - Min: 30.20, Max: 120.50, Mean: 103.59, Median: 105.00
2025-10-20 17:54:42.679 | WARNING  | __main__:<module>:15 - Found 6 shots with suspicious X coordinates (outside 0-120 range)
2025-10-20 17:54:42.681 | DEBUG    | __main__:<module>:16 - Sample of X outliers:
                   player_name                    team_name      x     y
19300        Konstantin Rausch                 Darmstadt 98  120.4  80.0
32604  Nicolae Claudiu Stanciu         

In [10]:
shots_df.loc[shots_df['x'] > 120, 'x'] = 120  # change to 120

In [11]:
# ============================================================================
# STEP 4: DISTANCE TO GOAL VALIDATION
# ============================================================================

logger.info("="*80)
logger.info("STEP 4: Validating Distance to Goal")
logger.info("="*80)

logger.info("Distance to goal statistics:")
logger.debug(f"\n{df['dist_to_goal'].describe()}")

# Check for unrealistic distances (max possible ~120m diagonally)
dist_outliers = df[df['dist_to_goal'] > 120]
if len(dist_outliers) > 0:
    logger.warning(f"Found {len(dist_outliers)} shots with distance > 120m (impossible on standard pitch)")
    logger.debug(f"Sample:\n{dist_outliers[['player_name', 'x', 'y', 'dist_to_goal']].head()}")
else:
    logger.success("All distances to goal are plausible")

# Check for zero or negative distances
zero_dist = df[df['dist_to_goal'] <= 0]
if len(zero_dist) > 0:
    logger.error(f"Found {len(zero_dist)} shots with distance <= 0 (invalid)")
    logger.debug(f"Sample:\n{zero_dist[['player_name', 'x', 'y', 'dist_to_goal']].head()}")

# Check if distance matches coordinates (goal typically at x=120, y=40)
# Recalculate distance assuming goal at (120, 40)
df['calculated_dist'] = np.sqrt((120 - df['x'])**2 + (40 - df['y'])**2)
dist_diff = abs(df['dist_to_goal'] - df['calculated_dist'])
large_discrepancies = df[dist_diff > 5]

if len(large_discrepancies) > 0:
    logger.warning(f"Found {len(large_discrepancies)} shots where reported distance differs from calculated distance by >5m")
    logger.debug(f"This might indicate different goal position assumptions or data errors")
else:
    logger.success("Distance values match coordinate-based calculations")

2025-10-20 17:54:53.970 | INFO     | __main__:<module>:5 - ================================================================================
2025-10-20 17:54:53.972 | INFO     | __main__:<module>:6 - STEP 4: Validating Distance to Goal
2025-10-20 17:54:53.972 | INFO     | __main__:<module>:7 - ================================================================================
2025-10-20 17:54:53.973 | INFO     | __main__:<module>:9 - Distance to goal statistics:
2025-10-20 17:54:54.004 | DEBUG    | __main__:<module>:10 - 
count    88023.000000
mean        19.180483
std          8.747756
min          0.400000
25%         12.000417
50%         18.427154
75%         25.495098
max         92.800862
Name: dist_to_goal, dtype: float64
2025-10-20 17:54:54.005 | SUCCESS  | __main__:<module>:18 - All distances to goal are plausible
2025-10-20 17:54:54.044 | SUCCESS  | __main__:<module>:36 - Distance values match coordinate-based calculations


In [12]:
shots_df.shot_outcome.value_counts()

shot_outcome
Off T               28466
Blocked             21677
Saved               20788
Goal                 9790
Wayward              4823
Post                 1842
Saved Off Target      349
Saved to Post         288
Name: count, dtype: int64

The field is considered to have 120x80 metres

In [19]:
shots_df[(shots_df['dist_to_goal'] > 60) & (shots_df['shot_outcome'] == 'Goal')]

,match_id,event_id,player_id,player_name,team_id,team_name,minute,second,shot_outcome,shot_body_part,shot_technique,x,y,dist_to_goal,angle_to_goal_rad,angle_to_goal_deg
26293,NaN,34c49775-c74a-4779-9bd5-6c266755bf69,24035,Álvaro Vázquez García,7283,Kerala Blasters,81,4,Goal,Right Foot,Normal,51.9,28.6,69.047592,0.114158,6.540791
70520,NaN,802522cd-c977-4c7d-adc0-bfbe4607783b,16383,Danique Kerkdijk,973,Bristol City WFC,34,57,Goal,Right Foot,Normal,65.8,78.6,66.540213,0.097972,5.613363


In [20]:
# ============================================================================
# STEP 5: ANGLE TO GOAL VALIDATION
# ============================================================================

logger.info("="*80)
logger.info("STEP 5: Validating Angle to Goal")
logger.info("="*80)

logger.info("Angle statistics (radians):")
logger.debug(f"\n{df['angle_to_goal_rad'].describe()}")

# Angles should be between 0 and π (0 and 180 degrees)
angle_outliers_rad = df[(df['angle_to_goal_rad'] < 0) | (df['angle_to_goal_rad'] > np.pi)]
if len(angle_outliers_rad) > 0:
    logger.warning(f"Found {len(angle_outliers_rad)} shots with angles outside 0-π radians range")
    logger.debug(f"Sample:\n{angle_outliers_rad[['player_name', 'x', 'y', 'angle_to_goal_rad', 'angle_to_goal_deg']].head()}")
else:
    logger.success("All angles (radians) are within valid range")

logger.info("Angle statistics (degrees):")
logger.debug(f"\n{df['angle_to_goal_deg'].describe()}")

angle_outliers_deg = df[(df['angle_to_goal_deg'] < 0) | (df['angle_to_goal_deg'] > 180)]
if len(angle_outliers_deg) > 0:
    logger.warning(f"Found {len(angle_outliers_deg)} shots with angles outside 0-180 degrees range")
    logger.debug(f"Sample:\n{angle_outliers_deg[['player_name', 'x', 'y', 'angle_to_goal_deg']].head()}")
else:
    logger.success("All angles (degrees) are within valid range")

# Check consistency between radians and degrees
angle_consistency = abs(df['angle_to_goal_deg'] - np.degrees(df['angle_to_goal_rad']))
inconsistent_angles = df[angle_consistency > 1]
if len(inconsistent_angles) > 0:
    logger.warning(f"Found {len(inconsistent_angles)} shots where radian/degree conversion is inconsistent")
    logger.debug(f"Sample:\n{inconsistent_angles[['angle_to_goal_rad', 'angle_to_goal_deg']].head()}")

2025-10-20 18:02:06.230 | INFO     | __main__:<module>:5 - ================================================================================
2025-10-20 18:02:06.241 | INFO     | __main__:<module>:6 - STEP 5: Validating Angle to Goal
2025-10-20 18:02:06.242 | INFO     | __main__:<module>:7 - ================================================================================
2025-10-20 18:02:06.244 | INFO     | __main__:<module>:9 - Angle statistics (radians):
2025-10-20 18:02:06.251 | DEBUG    | __main__:<module>:10 - 
count    88023.000000
mean         0.443590
std          0.275888
min          0.000000
25%          0.262407
50%          0.344910
75%          0.550534
max          3.141593
Name: angle_to_goal_rad, dtype: float64
2025-10-20 18:02:06.253 | SUCCESS  | __main__:<module>:18 - All angles (radians) are within valid range
2025-10-20 18:02:06.253 | INFO     | __main__:<module>:20 - Angle statistics (degrees):
2025-10-20 18:02:06.257 | DEBUG    | __main__:<module>:21 - 
count    88

In [ ]:
# ============================================================================
# STEP 6: TIME VALIDATION (MINUTE, SECOND)
# ============================================================================

logger.info("="*80)
logger.info("STEP 6: Validating Match Time")
logger.info("="*80)

logger.info("Minute statistics:")
logger.debug(f"\n{df['minute'].describe()}")

# Standard match is 90 minutes + potential extra time
unusual_minutes = df[df['minute'] > 120]
if len(unusual_minutes) > 0:
    logger.warning(f"Found {len(unusual_minutes)} shots after minute 120 (check for data errors)")
    logger.debug(f"Sample:\n{unusual_minutes[['player_name', 'minute', 'second']].head()}")

negative_minutes = df[df['minute'] < 0]
if len(negative_minutes) > 0:
    logger.error(f"Found {len(negative_minutes)} shots with negative minutes (invalid)")

logger.info("Second statistics:")
logger.debug(f"\n{df['second'].describe()}")

invalid_seconds = df[(df['second'] < 0) | (df['second'] >= 60)]
if len(invalid_seconds) > 0:
    logger.error(f"Found {len(invalid_seconds)} shots with seconds outside 0-59 range")
    logger.debug(f"Sample:\n{invalid_seconds[['player_name', 'minute', 'second']].head()}")
else:
    logger.success("All second values are valid (0-59)")

2025-10-20 18:02:32.114 | INFO     | __main__:<module>:5 - ================================================================================
2025-10-20 18:02:32.115 | INFO     | __main__:<module>:6 - STEP 6: Validating Match Time
2025-10-20 18:02:32.116 | INFO     | __main__:<module>:7 - ================================================================================
2025-10-20 18:02:32.117 | INFO     | __main__:<module>:9 - Minute statistics:
2025-10-20 18:02:32.125 | DEBUG    | __main__:<module>:10 - 
count    88023.000000
mean        49.136192
std         27.269793
min          0.000000
25%         26.000000
50%         49.000000
75%         72.000000
max        139.000000
Name: minute, dtype: float64
2025-10-20 18:02:32.133 | WARNING  | __main__:<module>:15 - Found 333 shots after minute 120 (check for data errors)
2025-10-20 18:02:32.139 | DEBUG    | __main__:<module>:16 - Sample:
                      player_name  minute  second
1959               Teboho Mokoena     121       9
19

In [23]:
# There are 333 shots happening over minute 120, we define them to be at minute 120
shots_df.loc[shots_df['minute'] > 120, 'minute'] = 120

In [25]:
# ============================================================================
# STEP 7: CATEGORICAL VARIABLES VALIDATION
# ============================================================================

logger.info("="*80)
logger.info("STEP 7: Validating Categorical Variables")
logger.info("="*80)

# Shot outcome
logger.info("Shot outcome distribution:")
outcome_counts = df['shot_outcome'].value_counts()
for outcome, count in outcome_counts.items():
    logger.debug(f"  - {outcome}: {count} ({count/len(df)*100:.1f}%)")

unusual_outcomes = df[~df['shot_outcome'].isin(['Goal', 'Saved', 'Off T', 'Blocked', 'Wayward', 'Post'])]
if len(unusual_outcomes) > 0:
    logger.warning(f"Found {len(unusual_outcomes)} shots with unusual outcomes")
    logger.debug(f"Unique unusual outcomes: {unusual_outcomes['shot_outcome'].unique()}")

# Shot body part
logger.info("Shot body part distribution:")
body_part_counts = df['shot_body_part'].value_counts()
for part, count in body_part_counts.items():
    logger.debug(f"  - {part}: {count} ({count/len(df)*100:.1f}%)")

unusual_body_parts = df[~df['shot_body_part'].isin(['Right Foot', 'Left Foot', 'Head', 'Other'])]
if len(unusual_body_parts) > 0:
    logger.warning(f"Found {len(unusual_body_parts)} shots with unusual body parts")
    logger.debug(f"Unique unusual body parts: {unusual_body_parts['shot_body_part'].unique()}")

# Shot technique
logger.info("Shot technique distribution:")
technique_counts = df['shot_technique'].value_counts()
for technique, count in technique_counts.items():
    logger.debug(f"  - {technique}: {count} ({count/len(df)*100:.1f}%)")


2025-10-20 18:07:39.737 | INFO     | __main__:<module>:5 - ================================================================================
2025-10-20 18:07:39.739 | INFO     | __main__:<module>:6 - STEP 7: Validating Categorical Variables
2025-10-20 18:07:39.740 | INFO     | __main__:<module>:7 - ================================================================================
2025-10-20 18:07:39.740 | INFO     | __main__:<module>:10 - Shot outcome distribution:
2025-10-20 18:07:39.746 | DEBUG    | __main__:<module>:13 -   - Off T: 28466 (32.3%)
2025-10-20 18:07:39.746 | DEBUG    | __main__:<module>:13 -   - Blocked: 21677 (24.6%)
2025-10-20 18:07:39.747 | DEBUG    | __main__:<module>:13 -   - Saved: 20788 (23.6%)
2025-10-20 18:07:39.747 | DEBUG    | __main__:<module>:13 -   - Goal: 9790 (11.1%)
2025-10-20 18:07:39.747 | DEBUG    | __main__:<module>:13 -   - Wayward: 4823 (5.5%)
2025-10-20 18:07:39.748 | DEBUG    | __main__:<module>:13 -   - Post: 1842 (2.1%)
2025-10-20 18:07:39.748 | 

In [31]:
shots_df=shots_df[shots_df['shot_outcome'].isin(['Goal', 'Saved', 'Off T', 'Blocked', 'Wayward', 'Post'])]

In [33]:
# ============================================================================
# STEP 8: DUPLICATE DETECTION
# ============================================================================

logger.info("="*80)
logger.info("STEP 8: Checking for Duplicates")
logger.info("="*80)

# Check for exact duplicates
exact_duplicates = df.duplicated().sum()
if exact_duplicates > 0:
    logger.warning(f"Found {exact_duplicates} exact duplicate rows")
else:
    logger.success("No exact duplicate rows found")

# Check for duplicates based on key fields (same event)
event_duplicates = df.duplicated(subset=['event_id'], keep=False).sum()
if event_duplicates > 0:
    logger.warning(f"Found {event_duplicates} rows with duplicate event_ids")
    logger.debug("This might indicate the same shot recorded multiple times")
else:
    logger.success("No duplicate event_ids found")


2025-10-20 18:10:58.752 | INFO     | __main__:<module>:5 - ================================================================================
2025-10-20 18:10:58.754 | INFO     | __main__:<module>:6 - STEP 8: Checking for Duplicates
2025-10-20 18:10:58.754 | INFO     | __main__:<module>:7 - ================================================================================
2025-10-20 18:10:58.808 | SUCCESS  | __main__:<module>:14 - No exact duplicate rows found
2025-10-20 18:10:58.815 | SUCCESS  | __main__:<module>:22 - No duplicate event_ids found


IQR method doesn't take context into account, so it is not useful here

In [ ]:
# # ============================================================================
# # STEP 9: STATISTICAL OUTLIERS (IQR METHOD)
# # ============================================================================

# logger.info("="*80)
# logger.info("STEP 9: Detecting Statistical Outliers (IQR Method)")
# logger.info("="*80)

# numeric_cols = ['x', 'y', 'dist_to_goal', 'angle_to_goal_deg']

# for col in numeric_cols:
#     Q1 = df[col].quantile(0.25)
#     Q3 = df[col].quantile(0.75)
#     IQR = Q3 - Q1
#     lower_bound = Q1 - 3 * IQR  # Using 3*IQR for extreme outliers
#     upper_bound = Q3 + 3 * IQR
    
#     outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    
#     if len(outliers) > 0:
#         logger.warning(f"{col}: Found {len(outliers)} statistical outliers (beyond 3*IQR)")
#         logger.debug(f"  Valid range: [{lower_bound:.2f}, {upper_bound:.2f}]")
#         logger.debug(f"  Outlier range: [{outliers[col].min():.2f}, {outliers[col].max():.2f}]")
#     else:
#         logger.success(f"{col}: No extreme statistical outliers found")

2025-10-20 18:11:17.283 | INFO     | __main__:<module>:5 - ================================================================================
2025-10-20 18:11:17.285 | INFO     | __main__:<module>:6 - STEP 9: Detecting Statistical Outliers (IQR Method)
2025-10-20 18:11:17.285 | INFO     | __main__:<module>:7 - ================================================================================
2025-10-20 18:11:17.292 | WARNING  | __main__:<module>:21 - x: Found 76 statistical outliers (beyond 3*IQR)
2025-10-20 18:11:17.292 | DEBUG    | __main__:<module>:22 -   Valid range: [57.40, 150.50]
2025-10-20 18:11:17.294 | DEBUG    | __main__:<module>:23 -   Outlier range: [30.20, 57.40]
2025-10-20 18:11:17.299 | SUCCESS  | __main__:<module>:25 - y: No extreme statistical outliers found
2025-10-20 18:11:17.305 | WARNING  | __main__:<module>:21 - dist_to_goal: Found 65 statistical outliers (beyond 3*IQR)
2025-10-20 18:11:17.305 | DEBUG    | __main__:<module>:22 -   Valid range: [-28.48, 65.98]
2025-10

Wrong values corrected for shots_df dataset

In [35]:
# ============================================================================
# STEP 10: SUMMARY AND RECOMMENDATIONS
# ============================================================================

logger.info("="*80)
logger.info("STEP 10: Summary and Cleaning Recommendations")
logger.info("="*80)

total_issues = 0
total_issues += len(x_outliers) + len(y_outliers)
total_issues += len(dist_outliers) + len(zero_dist)
total_issues += len(angle_outliers_rad) + len(angle_outliers_deg)
total_issues += len(invalid_seconds) + len(negative_minutes)
total_issues += exact_duplicates

logger.info(f"Total rows analyzed: {len(df)}")
logger.warning(f"Total potential issues found: {total_issues}")

logger.info("\n📋 Recommended cleaning steps:")
logger.info("1. Remove or investigate rows with invalid coordinates (outside pitch)")
logger.info("2. Fix or remove rows with impossible distances (>120m or <=0)")
logger.info("3. Correct angle values outside valid ranges")
logger.info("4. Fix invalid time values (negative minutes, seconds >=60)")
logger.info("5. Remove exact duplicate rows")
logger.info("6. Standardize categorical variable names (check for typos)")
logger.info("7. Handle missing values based on their percentage and importance")

logger.success("EDA completed! Check the log file for detailed information.")

# Optional: Create a cleaned dataset
logger.info("\nCreating cleaned dataset...")
df_clean = df.copy()

# Apply basic filters
df_clean = df_clean[
    (df_clean['x'] >= 0) & (df_clean['x'] <= 120) &
    (df_clean['y'] >= 0) & (df_clean['y'] <= 80) &
    (df_clean['dist_to_goal'] > 0) & (df_clean['dist_to_goal'] <= 120) &
    (df_clean['angle_to_goal_deg'] >= 0) & (df_clean['angle_to_goal_deg'] <= 180) &
    (df_clean['second'] >= 0) & (df_clean['second'] < 60) &
    (df_clean['minute'] >= 0)
]

# Remove duplicates
df_clean = df_clean.drop_duplicates()

rows_removed = len(df) - len(df_clean)
logger.info(f"Rows removed: {rows_removed} ({rows_removed/len(df)*100:.2f}%)")
logger.info(f"Clean dataset size: {len(df_clean)} rows")

# Save cleaned dataset
# df_clean.to_csv('shots_data_cleaned.csv', index=False)
# logger.success("Cleaned dataset saved to 'shots_data_cleaned.csv'")

2025-10-20 18:19:22.718 | INFO     | __main__:<module>:5 - ================================================================================
2025-10-20 18:19:22.725 | INFO     | __main__:<module>:6 - STEP 10: Summary and Cleaning Recommendations
2025-10-20 18:19:22.725 | INFO     | __main__:<module>:7 - ================================================================================
2025-10-20 18:19:22.726 | INFO     | __main__:<module>:16 - Total rows analyzed: 88023
2025-10-20 18:19:22.726 | WARNING  | __main__:<module>:17 - Total potential issues found: 6
2025-10-20 18:19:22.727 | INFO     | __main__:<module>:19 - 
📋 Recommended cleaning steps:
2025-10-20 18:19:22.727 | INFO     | __main__:<module>:20 - 1. Remove or investigate rows with invalid coordinates (outside pitch)
2025-10-20 18:19:22.727 | INFO     | __main__:<module>:21 - 2. Fix or remove rows with impossible distances (>120m or <=0)
2025-10-20 18:19:22.729 | INFO     | __main__:<module>:22 - 3. Correct angle values outside

In [36]:
shots_df.to_csv('xG/results/shots_after_eda.csv', index=False)

## Second iteration
There might be too many low quality shots

In [1]:
import pandas as pd

In [3]:
import os

# Go one level higher
os.chdir("..")

# Check current directory
print(os.getcwd())

/Users/rodrigo/football-data-analytics/football_xG


In [20]:
df=pd.read_csv("xG/results/shots_after_eda.csv")

In [21]:
df.iloc[0]

match_id                                              NaN
event_id             943d24cf-3f41-41d0-81b9-c7237c374426
player_id                                           32119
player_name                                 Leandro Greco
team_id                                               226
team_name                                   Hellas Verona
minute                                                  9
second                                                 41
shot_outcome                                        Saved
shot_body_part                                  Left Foot
shot_technique                                Half Volley
x                                                    94.5
y                                                    54.5
dist_to_goal                                     29.33428
angle_to_goal_rad                                0.237022
angle_to_goal_deg                               13.580382
Name: 0, dtype: object

In [27]:
df.shot_outcome.value_counts()

shot_outcome
Off T      28466
Blocked    21677
Saved      20788
Goal        9790
Wayward     4823
Post        1842
Name: count, dtype: int64

### 1) Test blocked shots. Eliminate clear super low quality shots

In [6]:
# Check blocked shots statistics
blocked_shots = df[df['shot_outcome'] == 'Blocked']
print(blocked_shots[['dist_to_goal', 'angle_to_goal_deg']].describe())

       dist_to_goal  angle_to_goal_deg
count  21677.000000       21677.000000
mean      20.761594          21.832081
std        7.408839          11.112898
min        1.702939           0.000000
25%       15.239751          14.937891
50%       20.833867          18.424104
75%       26.126615          24.898044
max       66.446369         180.000000


In [22]:
# Remove only the worst blocked shots
df_clean = df[~((df['shot_outcome'] == 'Blocked') & (
            (df['dist_to_goal'] > 40) |  
            (df['angle_to_goal_deg'] > 150)))]

In [23]:
df_clean.shape

(87299, 16)

### 2) Wayward shots

In [18]:
wayward_shots = df[df['shot_outcome'] == 'Wayward']

print("Wayward Shots Analysis:")
print("="*50)
print(wayward_shots[['dist_to_goal', 'angle_to_goal_deg', 'shot_technique']].describe())
print("\nDistance distribution:")
print(wayward_shots['dist_to_goal'].quantile([0.25, 0.5, 0.75, 0.9, 0.95]))
print("\nAngle distribution:")
print(wayward_shots['angle_to_goal_deg'].quantile([0.25, 0.5, 0.75, 0.9, 0.95]))

Wayward Shots Analysis:
       dist_to_goal  angle_to_goal_deg
count   4823.000000        4823.000000
mean      16.927080          29.281713
std        9.952995          16.114027
min        1.529706           0.000000
25%       10.000250          16.267608
50%       13.984277          26.174314
75%       22.161453          39.286233
max       92.800862         157.545469

Distance distribution:
0.25    10.000250
0.50    13.984277
0.75    22.161453
0.90    28.902353
0.95    33.422999
Name: dist_to_goal, dtype: float64

Angle distribution:
0.25    16.267608
0.50    26.174314
0.75    39.286233
0.90    51.629520
0.95    58.419817
Name: angle_to_goal_deg, dtype: float64


Surprise result: Most Wayward Shots Are From GOOD Positions! These should mostly be KEPT! They're valuable for your xG model because they show that even from good positions, shots can fail.

In [24]:
# Remove only truly extreme wayward shots
df_clean_v2 = df_clean[
    ~(
        (df_clean['shot_outcome'] == 'Wayward') & 
        ((df_clean['dist_to_goal'] > 45) | 
        (df_clean['angle_to_goal_deg'] > 150)))]

In [25]:
df_clean_v2.shape

(87215, 16)

### 3) General analysis